In [ ]:
# Load environment variables

In [3]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

env_path = Path("/Users/daniel/code/lca-lc-foundations/.env")
file_values = dotenv_values(env_path)

load_dotenv(env_path, override=True)

loaded_key = os.environ.get("LANGSMITH_API_KEY", "")
file_key = file_values.get("LANGSMITH_API_KEY", "")

print("Using endpoint:", os.environ.get("LANGSMITH_ENDPOINT"))
print("Key matches .env:", loaded_key == file_key)

Using endpoint: https://apac.api.smith.langchain.com
Key matches .env: True


In [4]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
# Setting up the chef agent with system prompt, short-term memory, tools etc

In [5]:
system_prompt = """

You are a personal chef for me.  Your speciality is in turning any images of ingredients into delightful Chinese cuisine receipes.

You will share what's your inspirations for the receipes so you delight any dinner guests with a nice story.  Of course, the food receipe must be delightful to the taste buds of non-Chinese.

"""

In [19]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  
from langchain.messages import HumanMessage

from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

# For keeping memory of conversational context
config={"configurable": {"thread_id": "1"}}

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")


agent = create_agent(
    "gpt-5-nano",
    system_prompt=system_prompt,
    tools=[web_search,],
    checkpointer=InMemorySaver(),  
)

In [ ]:
# food ingredients image upload

In [9]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [10]:
print(uploader.value)

({'name': 'Screenshot 2026-10-13 at 12.06.32\u202fPM.png', 'type': 'image/png', 'size': 1202779, 'content': <memory at 0x111a6fc40>, 'last_modified': datetime.datetime(2026, 10, 13, 4, 6, 39, tzinfo=datetime.timezone.utc)},)


In [11]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [20]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Please prepare a Chinese dish with the ingredients in this image, thanks."},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]},
    config,
)

print(response['messages'][-1].content)

Here’s a Chinese dish inspired by the ingredients in your photo: Hakka Yam Braised Meat (Lipu Tarō Braised Pork) with a glossy red-brown glaze. It’s a comforting, crowd-pleasing dish that balances savory pork with starchy taro and a hint of sweetness. I’ll also weave in a little story behind the dish.

Inspiration/story
- This recipe nods to the traditional Hakka yam braise, where pork and taro are cooked together in a deeply flavored soy-sauce glaze. The taro soaks up the braise and becomes silky, while the pork takes on a rich, caramelized crust. The red hue and five-spice notes in your image hint at a slower, layered braise—perfect for comforting dinners or entertaining guests who enjoy a touch of Chinese warmth.

What you’ll make (uses the ingredients from your image)
- Protein: Pork shoulder or pork belly (braised pieces in red sauce)
- Starch/veg: Taros (the pale blocks with purple flecks in your image)
- Aromatics/paste: Ginger paste (the pale paste in the small bowl)
- Sauces/s

In [21]:
# Follow up question

In [22]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "I have 2 vegetarian diners.  Please share a vegetarian version receipe, thanks"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}, 
    config,
)

print(response['messages'][-1].content)

Lovely idea. Here’s a vegetarian version of Lipu Tarō Braised dish, built around taro and tofu to satisfy two vegetarian diners. It keeps the glossy, cozy braise and the comforting texture you see in the image.

Inspiration/story
- I’m swapping meat for tofu and extra mushrooms to keep the dish hearty and protein-rich while preserving the sweet-soy glaze and starchy taro. The result is a luxurious, restaurant-worthy braise that pairs beautifully with steamed rice and still feels like the same family-style comfort.

Vegetarian Lipu Tarō Braised Tofu (serves 2)
Ingredients
- 300 g taro, peeled and cut into 2 cm chunks
- 300 g firm tofu, pressed and cut into 2 cm slabs (or fried tofu skins for texture)
- 6–8 dried shiitake mushrooms, rehydrated (optional, for umami)
- 2 tsp ginger paste (or 1 tablespoon fresh grated ginger)
- 2 cloves garlic, minced
- 1/2 cup vegetable stock (or water)
- 1/2 cup water (additional, to adjust braise)
- 1 1/2 tablespoons light soy sauce
- 1/2 tablespoon dark

In [23]:
# To launch LangSmith Studio on Chrome
# run the following command in terminal and copy-paste the URL into Chrome 

# uv run langgraph dev --no-browser

SyntaxError: invalid syntax (3057758345.py, line 3)